# Evaluate TorchXRayVision models in PadChest dataset using balanced subsets of data for each pathology 

Evaluate 2 models available in TorchXRayVision Library:

1. resnet50: Trained in all datasets, Resolution = 512 x 512

2. densenet121: Trained in PadChest, Resolution = 224 x 224

---------------------------------
Changes in the class PC_Dataset (adapted from TorchXRayVision): 

1. Use "fillna" instead of removing empty labels: self.csv ['Labels'].fillna('', inplace = True)
        
2. unique_patients = False

3. Remove the age filter
---------------------------------

In [2]:
import os
import pandas as pd
import numpy as np
import torch
import tqdm
import sklearn
from sklearn import metrics
import torchvision
import torchxrayvision as xrv
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from typing import Dict, List
from skimage.io import imread
import random

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {DEVICE}')
import train_utils

Device: cuda


# Load model

### resnet50

In [3]:
model_resnet = xrv.models.ResNet(weights="resnet50-res512-all")
model_resnet.to(DEVICE)

XRV-ResNet-resnet50-res512-all

### densenet121

In [4]:
model_densenet = xrv.models.DenseNet(weights="densenet121-res224-pc")
model_densenet.to(DEVICE)

XRV-DenseNet121-densenet121-res224-pc

# Create PC dataset

In [5]:
class Dataset:
    """The datasets in this library aim to fit a simple interface where the
    imgpath and csvpath are specified. Some datasets require more than one
    metadata file and for some the metadata files are packaged in the library
    so only the imgpath needs to be specified.
    """
    def __init__(self):
        pass

    pathologies: List[str]

    labels: np.ndarray

    csv: pd.DataFrame

    def totals(self) -> Dict[str, Dict[str, int]]:
        """Compute counts of pathologies.

        Returns: A dict containing pathology name -> (label->value)
        """
        counts = [dict(collections.Counter(items[~np.isnan(items)]).most_common()) for items in self.labels.T]
        return dict(zip(self.pathologies, counts))

    def __repr__(self) -> str:
        """Returns the name and a description of the dataset such as:

        .. code-block:: python

            CheX_Dataset num_samples=191010 views=['PA', 'AP']

        If in a jupyter notebook it will also print the counts of the
        pathology counts returned by .totals()

        .. code-block:: python

            {'Atelectasis': {0.0: 17621, 1.0: 29718},
             'Cardiomegaly': {0.0: 22645, 1.0: 23384},
             'Consolidation': {0.0: 30463, 1.0: 12982},
             ...}

        """
        if xrv.utils.in_notebook():
            pprint.pprint(self.totals())
        return self.string()

    def check_paths_exist(self):
        if not os.path.isdir(self.imgpath):
            raise Exception("imgpath must be a directory")
        if not os.path.isfile(self.csvpath):
            raise Exception("csvpath must be a file")

    def limit_to_selected_views(self, views):
        """This function is called by subclasses to filter the
        images by view based on the values in .csv['view']
        """
        if type(views) is not list:
            views = [views]
        if '*' in views:
            # if you have the wildcard, the rest are irrelevant
            views = ["*"]
        self.views = views

        # missing data is unknown
        self.csv.view.fillna("UNKNOWN", inplace=True)

        if "*" not in views:
            self.csv = self.csv[self.csv["view"].isin(self.views)]  # Select the view

In [6]:
def normalize(img, maxval, reshape=False):
    """Scales images to be roughly [-1024 1024].

    Call xrv.utils.normalize moving forward.
    """
    return xrv.utils.normalize(img, maxval, reshape)


def apply_transforms(sample, transform, seed=None) -> Dict:
    """Applies transforms to the image and masks.
    The seeds are set so that the transforms that are applied
    to the image are the same that are applied to each mask.
    This way data augmentation will work for segmentation or 
    other tasks which use masks information.
    """

    if seed is None:
        MAX_RAND_VAL = 2147483647
        seed = np.random.randint(MAX_RAND_VAL)

    if transform is not None:
        random.seed(seed)
        torch.random.manual_seed(seed)
        sample["img"] = transform(sample["img"])

        if "pathology_masks" in sample:
            for i in sample["pathology_masks"].keys():
                random.seed(seed)
                torch.random.manual_seed(seed)
                sample["pathology_masks"][i] = transform(sample["pathology_masks"][i])

        if "semantic_masks" in sample:
            for i in sample["semantic_masks"].keys():
                random.seed(seed)
                torch.random.manual_seed(seed)
                sample["semantic_masks"][i] = transform(sample["semantic_masks"][i])

    return sample

In [19]:
class PC_Dataset(Dataset):

    def __init__(self,
                 imgpath,
                 csvpath,
                 avail_imgID,
                 views=["PA"],
                 transform=None,
                 data_aug=None,
                 flat_dir=True,
                 seed=0,
                 unique_patients=True
                 ):

        super(PC_Dataset, self).__init__()
        np.random.seed(seed)  # Reset the seed so all runs are the same.

        self.pathologies = ["Atelectasis", "Consolidation", "Infiltration",
                            "Pneumothorax", "Edema", "Emphysema", "Fibrosis",
                            "Effusion", "Pneumonia", "Pleural_Thickening",
                            "Cardiomegaly", "Nodule", "Mass", "Hernia", "Fracture",
                            "Granuloma", "Flattened Diaphragm", "Bronchiectasis",
                            "Aortic Elongation", "Scoliosis",
                            "Hilar Enlargement", "Tuberculosis",
                            "Air Trapping", "Costophrenic Angle Blunting", "Aortic Atheromatosis",
                            "Hemidiaphragm Elevation",
                            "Support Devices", "Tube'"]  # the Tube' is intentional

        self.pathologies = sorted(self.pathologies)

        mapping = dict()

        mapping["Infiltration"] = ["infiltrates",
                                   "interstitial pattern",
                                   "ground glass pattern",
                                   "reticular interstitial pattern",
                                   "reticulonodular interstitial pattern",
                                   "alveolar pattern",
                                   "consolidation",
                                   "air bronchogram"]
        mapping["Pleural_Thickening"] = ["pleural thickening"]
        mapping["Consolidation"] = ["air bronchogram"]
        mapping["Hilar Enlargement"] = ["adenopathy",
                                        "pulmonary artery enlargement"]
        mapping["Support Devices"] = ["device",
                                      "pacemaker"]
        mapping["Tube'"] = ["stent'"]  # the ' is to select findings which end in that word

        self.imgpath = imgpath
        self.transform = transform
        self.data_aug = data_aug
        self.flat_dir = flat_dir
        self.csvpath = csvpath
        
        self.csv = pd.read_csv(self.csvpath, low_memory=False)

        # Standardize view names
        self.csv.loc[self.csv["Projection"].isin(["AP_horizontal"]), "Projection"] = "AP Supine"

        self.csv["view"] = self.csv['Projection']
        self.limit_to_selected_views(views)
        
        #----------------ADDED------------------
        # Remove null stuff
        #self.csv = self.csv[~self.csv["Labels"].isnull()]
        self.csv['Labels'].fillna('', inplace = True)
        
        # Filter by available image files
        self.csv = self.csv[self.csv["ImageID"].isin(avail_imgID)]
        
        # Exclude the labels:
        self.csv = self.csv[~self.csv["Labels"].str.contains('exclude')]
        self.csv = self.csv[~self.csv["Labels"].str.contains('suboptimal study')]
        #----------------ADDED------------------
        

        #----------------Not used in this case------------------
        if unique_patients:
            self.csv = self.csv.groupby("PatientID").first().reset_index()

        # Filter out age < 10 (paper published 2019)
        #self.csv = self.csv[(2019 - self.csv.PatientBirth > 10)]
        
        #------------------------------------------------------

        # Get our classes.
        labels = []
        for pathology in self.pathologies:
            mask = self.csv["Labels"].str.contains(pathology.lower())
            if pathology in mapping:
                for syn in mapping[pathology]:
                    #print("mapping", syn)
                    mask |= self.csv["Labels"].str.contains(syn.lower())
            labels.append(mask.values)
        self.labels = np.asarray(labels).T
        self.labels = self.labels.astype(np.float32)

        self.pathologies[self.pathologies.index("Tube'")] = "Tube"

        # add consistent csv values

        # offset_day_int
        dt = pd.to_datetime(self.csv["StudyDate_DICOM"], format="%Y%m%d")
        self.csv["offset_day_int"] = dt.astype(np.int64) // 10**9 // 86400

        # patientid
        self.csv["patientid"] = self.csv["PatientID"].astype(str)

        # age
        self.csv['age_years'] = (2017 - self.csv['PatientBirth'])

        # sex
        self.csv['sex_male'] = self.csv['PatientSex_DICOM'] == 'M'
        self.csv['sex_female'] = self.csv['PatientSex_DICOM'] == 'F'

    def string(self):
        return self.__class__.__name__ + " num_samples={} views={} data_aug={}".format(len(self), self.views, self.data_aug)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        sample = {}
        sample["idx"] = idx
        sample["lab"] = self.labels[idx]

        imgid = self.csv['ImageID'].iloc[idx]
        img_path = os.path.join(self.imgpath, imgid)
        img = imread(img_path)

        sample["img"] = normalize(img, maxval=65535, reshape=True)

        sample = apply_transforms(sample, self.transform)
        sample = apply_transforms(sample, self.data_aug)

        return sample

## Define the function to process the data

In [23]:
def process_data(dataset, model):
    outs = []
    labs = []
    with torch.no_grad():
        np.random.seed(0)
        for i in tqdm.tqdm(range(len(dataset))):
            sample = dataset[i]
            labs.append(sample["lab"])
            image = torch.from_numpy(sample["img"]).unsqueeze(0)
            if torch.cuda.is_available():
                image = image.cuda()
            out = model(image).cpu()
            outs.append(out.detach().numpy()[0])
    
    return outs, labs

## Define the function to obtain the metrics

In [24]:
def calculate_metrics(index, outs, labs):
    result = {}
    result["Pathology"] = model_resnet.pathologies[index]
    if len(np.unique(np.asarray(labs)[:, index])) > 1:
        labels = np.asarray(labs)[:, index].astype(bool)
        preds = np.asarray(outs)[:, index]
        result["AUC"] = sklearn.metrics.roc_auc_score(labels, preds)
        result["Acc"] = sklearn.metrics.accuracy_score(labels, preds > 0.5)
        result["Prec"] = sklearn.metrics.precision_score(labels, preds > 0.5)
        result["Recall"] = sklearn.metrics.recall_score(labels, preds > 0.5)
        result["F1"] = sklearn.metrics.f1_score(labels, preds > 0.5)
    return result

## Filter by available image IDs:  the txt file is different to evaluate each pathology

In [108]:
imageIDs_available = os.path.join("C:/Users/maria/OneDrive/Documentos/Escritorio/pathologies_imageIDs", "fracture_IDs.txt")
with open(imageIDs_available, "r") as f:
    avail_ID = [line.strip() for line in f.readlines()]

# Evaluate the model: Resnet 50

### Load the dataset

In [109]:
d_pc = PC_Dataset(imgpath = "C:/Users/maria/OneDrive/Documentos/Escritorio/CodeChestXRay/TrainXray/Data/PadChest512_PA_Manually_Filtered",
                 csvpath = "C:/Users/maria/OneDrive/Documentos/Escritorio/CodeChestXRay/TrainXray/Data/PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv" ,
                 avail_imgID = avail_ID, unique_patients = False )
xrv.datasets.relabel_dataset(xrv.datasets.default_pathologies, d_pc)

{'Tuberculosis', 'Hemidiaphragm Elevation', 'Aortic Atheromatosis', 'Costophrenic Angle Blunting', 'Aortic Elongation', 'Tube', 'Support Devices', 'Scoliosis', 'Flattened Diaphragm', 'Air Trapping', 'Granuloma', 'Bronchiectasis', 'Hilar Enlargement'} will be dropped
Lung Lesion doesn't exist. Adding nans instead.
Lung Opacity doesn't exist. Adding nans instead.
Enlarged Cardiomediastinum doesn't exist. Adding nans instead.


In [110]:
len(d_pc)

5048

## Using optimal threshold: op_threshs

In [13]:
print("Optimal thresholds for each pathology:\n", model_resnet.op_threshs)
print("\nApply sigmoid:\n", model_resnet.apply_sigmoid)

Optimal thresholds for each pathology:
 tensor([0.5157, 0.5044, 0.5379, 0.5072, 0.5025, 0.5035, 0.5038, 0.5186, 0.5078,
        0.5072, 0.5056, 0.5107, 0.5054, 0.5021,    nan, 0.5081, 0.5139,    nan],
       device='cuda:0')

Apply sigmoid:
 False


### Atelectasis

In [15]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 0 directly
results.append(calculate_metrics(0,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 7676/7676 [06:37<00:00, 19.32it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Atelectasis,0.793058,0.512507,0.980198,0.025788,0.050254


### Infiltrations

In [33]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 0 directly
results.append(calculate_metrics(2,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████| 15006/15006 [12:35<00:00, 19.87it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Infiltration,0.83206,0.532254,0.956604,0.067573,0.126229


### pneumothorax

In [38]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 3 directly
results.append(calculate_metrics(3,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████████| 434/434 [00:21<00:00, 20.49it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Pneumothorax,0.82703,0.506912,1.0,0.013825,0.027273


### effusion

In [42]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 7 directly
results.append(calculate_metrics(7,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 6492/6492 [05:24<00:00, 20.00it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Effusion,0.949035,0.657579,0.99325,0.317314,0.480971


### pneumonia

In [48]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 8 directly
results.append(calculate_metrics(8,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 6614/6614 [05:30<00:00, 20.03it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Pneumonia,0.778477,0.515271,0.963303,0.031751,0.061475


### cardiomegaly

In [52]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 10 directly
results.append(calculate_metrics(10,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████| 16228/16228 [13:32<00:00, 19.97it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Cardiomegaly,0.931681,0.569756,0.978849,0.142593,0.248924


### Nodule

In [57]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 11 directly
results.append(calculate_metrics(11,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 7080/7080 [06:02<00:00, 19.52it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Nodule,0.760679,0.520621,0.967949,0.042655,0.08171


### Mass

In [63]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 12 directly
results.append(calculate_metrics(12,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 1566/1566 [01:18<00:00, 19.98it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Mass,0.857022,0.559387,0.960396,0.123883,0.219457


### Facture

In [67]:
outs, labs = process_data(d_pc, model_resnet)
results = []

# Calculate metrics for element 15 directly
results.append(calculate_metrics(15,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 5048/5048 [04:14<00:00, 19.86it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Fracture,0.863532,0.610935,0.9811,0.226228,0.367675


# Evaluate the model: Densenet 121

In [68]:
print("Optimal thresholds for each pathology:\n", model_densenet.op_threshs)
print("\nApply sigmoid:\n", model_densenet.apply_sigmoid)

Optimal thresholds for each pathology:
 tensor([0.0310, 0.0133, 0.0814, 0.0013, 0.0026, 0.0036, 0.0023, 0.0554, 0.0444,
        0.0428, 0.0433, 0.0376, 0.0057, 0.0092,    nan, 0.0265,    nan,    nan],
       device='cuda:0')

Apply sigmoid:
 False


### Atelectasis

In [73]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 0 directly
results.append(calculate_metrics(0,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 7678/7678 [09:38<00:00, 13.28it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Atelectasis,0.78984,0.713597,0.686448,0.786403,0.733034


### Infiltrations

In [77]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 0 directly
results.append(calculate_metrics(2,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████| 15006/15006 [19:09<00:00, 13.05it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Infiltration,0.844848,0.760029,0.739328,0.803279,0.769978


### pneumothorax

In [81]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 3 directly
results.append(calculate_metrics(3,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████████| 434/434 [00:32<00:00, 13.32it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Pneumothorax,0.90107,0.824885,0.794979,0.875576,0.833333


### effusion

In [85]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 7 directly
results.append(calculate_metrics(7,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 6492/6492 [08:22<00:00, 12.91it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Effusion,0.951487,0.886784,0.890513,0.882009,0.886241


### pneumonia

In [89]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 8 directly
results.append(calculate_metrics(8,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 6614/6614 [08:21<00:00, 13.18it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Pneumonia,0.849298,0.77094,0.769717,0.773208,0.771459


### cardiomegaly

In [93]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 10 directly
results.append(calculate_metrics(10,outs, labs))

pd.DataFrame(results)

100%|████████████████████████████████████████████████████████████████████████████| 16228/16228 [19:29<00:00, 13.87it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Cardiomegaly,0.931356,0.85981,0.834001,0.898447,0.865025


### nodule

In [102]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 11 directly
results.append(calculate_metrics(11,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 7080/7080 [08:47<00:00, 13.43it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Nodule,0.680734,0.62387,0.602191,0.729944,0.659941


### mass

In [97]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 12 directly
results.append(calculate_metrics(12,outs, labs))

pd.DataFrame(results)

100%|██████████████████████████████████████████████████████████████████████████████| 1566/1566 [02:02<00:00, 12.77it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Mass,0.85597,0.783525,0.792876,0.767561,0.780013


### fracture

In [69]:
outs, labs = process_data(d_pc, model_densenet)
results = []

# Calculate metrics for element 15 directly
results.append(calculate_metrics(15,outs, labs))

pd.DataFrame(results)

  0%|                                                                                 | 1/5048 [00:00<11:49,  7.12it/s]

100%|██████████████████████████████████████████████████████████████████████████████| 5048/5048 [06:30<00:00, 12.93it/s]


,Pathology,AUC,Acc,Prec,Recall,F1
0,Fracture,0.718549,0.664422,0.646127,0.727021,0.684191
